In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys
from pathlib import Path
from torch.utils.data import DataLoader

join = os.path.join
import torch

from skimage import io, transform
import torch.nn.functional as F

from pytorch_ood.detector import Mahalanobis
from pytorch_ood.utils import OODMetrics

import monai
from monai.metrics import DiceMetric

from PIL import Image

from peft import LoraConfig, get_peft_model

/export/home/rstanciu/FM_thesis_Razvan/FM_thesis/VersaMammo/downstream/.venv-versamammo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
<frozen importlib._bootstrap_external>:1297: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [2]:
PROJECT_ROOT=Path("/export/home/rstanciu/FM_thesis_Razvan/FM_thesis/")

MEDSAM_IMAGE_SIZE=(1024,1024)
DATA_ROOT=Path("/mnt/data/spathak")
CSV_DATA_PATH= DATA_ROOT / "CLaM-Annot-metadata.csv"
WEIGHTS_PATH= PROJECT_ROOT / "pipelines_and_experiments/lora_medsam_from_zgt_gt_component_boxes/ZGT_VersaMammo_fold4/weights"
MEDSAM_ROOT= PROJECT_ROOT / "MedSAM"
MEDSAM_CHECKPOINT = MEDSAM_ROOT / "work_dir" / "medsam_vit_b.pth"
OUTPUT_FEATURES=PROJECT_ROOT / "feature_space_inspection"

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

In [3]:
for path in [PROJECT_ROOT, MEDSAM_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))



from utils.dep_injection_util import (
        BaseDataset,
        NpzLoader,
        PngLoader,
        ConnectedComponentsBBoxFromMask
)

from MedSAM.segment_anything import sam_model_registry
from MedSAM.utils.medSAM_architecture import MedSAM, MedSAMPreprocess


In [4]:
df=pd.read_csv(CSV_DATA_PATH, delimiter=";")
df['AbnormalityType']=df['AbnormalityType'].fillna("None")
df['ImagePath']=df['ImagePath'].fillna("None")
df['ROIPath']=df['ROIPath'].fillna("None")

df_masses=df[df['AbnormalityType']=="Mass"].reset_index()
df_no_masses=df[df['AbnormalityType']!="Mass"].reset_index()

In [5]:
ckpt_path = WEIGHTS_PATH / "medsam_model_best.pth"

sam_model = sam_model_registry["vit_b"](checkpoint=MEDSAM_CHECKPOINT)

model = MedSAM(
    image_encoder=sam_model.image_encoder,
    mask_decoder=sam_model.mask_decoder,
    prompt_encoder=sam_model.prompt_encoder,
).to(DEVICE)


config = LoraConfig(
    r=8,
    lora_alpha=8,
    target_modules=["qkv"],
    lora_dropout=0.1,
    bias="none"
)
model = get_peft_model(model, config)


checkpoint = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(checkpoint["model"])

model.eval()

PeftModel(
  (base_model): LoraModel(
    (model): MedSAM(
      (image_encoder): ImageEncoderViT(
        (patch_embed): PatchEmbed(
          (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        )
        (blocks): ModuleList(
          (0-11): 12 x Block(
            (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True, bias=True)
            (attn): Attention(
              (qkv): lora.Linear(
                (base_layer): Linear(in_features=768, out_features=2304, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): Param

In [6]:

def collect_ZGT_files(root, metadata_df):
    root = Path(root) if isinstance(root, str) else root

    samples = []
    for row in metadata_df.itertuples():
        image_file= root / Path(row.ImagePath)
        mask_file=  root / Path(row.ROIPath) if Path(row.ROIPath).name != "None" else None

        samples.append({
            "image_path": image_file,
            "mask_path": mask_file,
        })  

    return samples

In [7]:
loader = PngLoader(dtype=np.uint8)

bbox_generator = ConnectedComponentsBBoxFromMask(
    annotation_threshold=0.5,
    allow_empty_mask=True,
)


id_dataset = BaseDataset(
    root=DATA_ROOT,
    format_loader=loader,
    file_collector=collect_ZGT_files,
    collector_kwargs={"metadata_df": df_masses},
    bbox_generator=bbox_generator,
    transforms=MedSAMPreprocess(MEDSAM_IMAGE_SIZE)
)



ood_dataset = BaseDataset(
    root=DATA_ROOT,
    format_loader=loader,
    file_collector=collect_ZGT_files,
    collector_kwargs={"metadata_df": df_no_masses},
    bbox_generator=bbox_generator,
    transforms=MedSAMPreprocess(MEDSAM_IMAGE_SIZE),
    label=-1
)





In [8]:
# Pass ID data through encoder and fit Mahalanobis distance metric
feats_list_id = []
labels_list_id = []


with torch.no_grad():
    for sample in id_dataset:
        imgs = sample["image"].to(DEVICE).unsqueeze(0)   #unsqueeze only if this is not using the batch loader
        y = sample["label"].to(DEVICE)

        z = model.image_encoder(imgs)              # [B, C, H, W] feature maps
        z = z.mean(dim=(2, 3))                     # [B, C] global average pool

        feats_list_id.append(z)
        labels_list_id.append(y)

Z_id = torch.cat(feats_list_id, dim=0)
Y_id = torch.stack(labels_list_id)

detector = Mahalanobis(encoder=None)
detector.fit_features(Z_id, Y_id)

In [9]:
torch.save(Z_id.detach().cpu(), OUTPUT_FEATURES / "medSAM_LoRA_ZGT_ID_allmasses")

In [10]:
# Predict on OOD dataset
feats_list = []

with torch.no_grad():
    for batch in ood_dataset:
        imgs = batch["image"].to(DEVICE).unsqueeze(0)
        z = model.image_encoder(imgs)
        z = z.mean(dim=(2, 3))
        feats_list.append(z)

Z_ood = torch.cat(feats_list, dim=0)


torch.save(Z_ood.detach().cpu(), OUTPUT_FEATURES / "medSAM_LoRA_ZGT_OOD_nomasses")

scores_id = detector.predict_features(Z_id)
scores_ood = detector.predict_features(Z_ood)

In [11]:
# 1) Make label tensors (same length as scores)
y_id  = torch.zeros(len(scores_id), dtype=torch.long)        # ID label = 0
y_ood = -torch.ones(len(scores_ood), dtype=torch.long)       # OOD label = -1

# 2) Concatenate
scores = torch.cat([scores_id.detach().cpu(), scores_ood.detach().cpu()], dim=0)
y_true = torch.cat([y_id, y_ood], dim=0)

# 3) Compute metrics
metrics = OODMetrics()
metrics.update(scores, y_true)
result = metrics.compute()

print(result)

{'AUROC': 0.9386181831359863, 'AUTC': 0.45504188537597656, 'AUPR-IN': 0.8117876648902893, 'AUPR-OUT': 0.9767126441001892, 'FPR95TPR': 0.5849056839942932}
